# 🏛️ CivicLens - Municipal AI Vision Model Training (Google Colab)
This notebook trains a lightweight, high-accuracy **MobileNetV3** model on the CivicLens PCMC dataset across 4 municipal classes:
- `pothole`: Road craters, cavities, asphalt damage
- `non_pothole`: Smooth, clean, intact road surfaces
- `garbage`: Uncollected waste, litter, overflowing bins
- `non_garbage`: Clean sidewalks, swept public spaces

**Hardware Requirement:** Free T4 GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# 1. Verify GPU is active
!nvidia-smi

In [ ]:
# 2. Unzip the dataset
# Upload your dataset zip (e.g., dataset_colab.zip) to Colab files in the left sidebar, then run this:
import os
import zipfile

zip_name = 'dataset_colab.zip' if os.path.exists('dataset_colab.zip') else 'dataset.zip'
if os.path.exists(zip_name):
    print(f"Unzipping {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('dataset_raw')
    print("Unzip complete!")
else:
    print(f"⚠️ Please upload {zip_name} into Colab files sidebar first!")

In [ ]:
# 3. Verify dataset directory structure
import glob
from pathlib import Path

# Find root folder that contains the 4 classes
possible_roots = ['dataset_raw', 'dataset_raw/dataset_colab', 'dataset_raw/dataset']
data_dir = 'dataset_raw'
for r in possible_roots:
    if os.path.exists(os.path.join(r, 'pothole')):
        data_dir = r
        break

print(f"Dataset root: {data_dir}")
classes = ['garbage', 'non_garbage', 'non_pothole', 'pothole']
for c in classes:
    count = len(glob.glob(f"{data_dir}/{c}/*.*"))
    print(f"  • {c:14}: {count} images found")

In [ ]:
# 4. PyTorch Data Loaders & Data Augmentations
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
print(f"Total dataset size: {len(full_dataset)} images")
print(f"Classes detected: {full_dataset.classes}")

# 80% Train, 20% Validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(42))

# Apply validation transform to val set
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

In [ ]:
# 5. Build MobileNetV3 Transfer Learning Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on compute device: {device}")

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
num_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(128, len(full_dataset.classes))
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

In [ ]:
# 6. Training & Validation Loop
import time

EPOCHS = 6
best_val_acc = 0.0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
    
    scheduler.step()
    train_acc = (train_correct / train_size) * 100
    train_loss = train_loss / train_size

    # Validation
    model.eval()
    val_loss, val_correct = 0.0, 0
    defect_correct, total_defect_tested = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()

            # Binary Defect vs Clean Check
            # classes: ['garbage', 'non_garbage', 'non_pothole', 'pothole']
            for p, l in zip(preds, labels):
                pred_class = full_dataset.classes[p.item()]
                label_class = full_dataset.classes[l.item()]
                is_pred_defect = pred_class in ['pothole', 'garbage']
                is_label_defect = label_class in ['pothole', 'garbage']
                if is_pred_defect == is_label_defect:
                    defect_correct += 1
                total_defect_tested += 1

    val_acc = (val_correct / val_size) * 100
    val_loss = val_loss / val_size
    defect_acc = (defect_correct / total_defect_tested) * 100

    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {train_loss:.4f} Acc: {train_acc:.1f}% | Val Loss: {val_loss:.4f} Val Acc: {val_acc:.1f}% | Defect Detection Acc: {defect_acc:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'classes': full_dataset.classes,
            'architecture': 'mobilenet_v3_small',
            'img_size': 224,
            'best_val_acc': best_val_acc
        }, 'civiclens_model.pt')

print(f"\n✓ Training finished in {round(time.time() - start_time, 1)} seconds!")
print(f"✓ Highest Validation Accuracy: {best_val_acc:.2f}%")
print("✓ Model checkpoint saved as: civiclens_model.pt")

In [ ]:
# 7. Download the trained model file to your PC
from google.colab import files
files.download('civiclens_model.pt')